Adım 1: Veri tabanı ve tablo bağlantısı

Adım 2: Ham EKG sinyallerinin okunması

Adım 3: Sinyal filtreleme ve temizleme

Adım 4: Kalite kontrol (Varyans filtresi)

Adım 5: R-Tepe (R-Peak) tespiti

Adım 6: Siyam ağı için hizalama (Segmentation)

Adım 7: Genlik normalizasyonu (Min-Max Scaling)

Adım 8: ThreadPool ile paralel işlem (Multitasking)

Adım 9: Kalıcı depolama (NumPy Save)

# TODO:
"""
🚀 Neden Bu Mimari Çok Başarılı?
Güvenlik Katmanlaşması:

Model A (0.8+): Senin "Altın Standardın". En yüksek hassasiyete sahip. Kimlik doğrulama için kullanılacak ana model.

Model B (0.5 - 0.8): "Esnek Model". Gürültüyü tolere edebilen, ancak Model A kadar keskin olmayan ara model.

Model C (0.5-): Bu grup bir "Eğitim seti" değil, bir "Reddetme Seti" olmalı. Bu seviyedeki sinyalleri eğitmek, sistemi yanıltmaya (false acceptance) neden olabilir.

Jüriyi Büyüleyecek "Akıllı Geçiş" Mekanizması:
Sistemine bir "Decision Engine" (Karar Mekanizması) ekleyebilirsin:

Kullanıcı EKG'sini gönderdi.

Önce nk.ecg_quality() çalışır.

Skor 0.8'den büyükse Model A'ya yönlendirilir.

Skor 0.5 - 0.8 arasındaysa Model B'ye yönlendirilir.

Skor 0.5'ten düşükse sistem kullanıcıya "Sinyal çok zayıf, lütfen saatini sıkılaştır ve hareketsiz kal" uyarısı verir.

🛠️ İzleyeceğin Yol Haritası
Bu mimariyi kurmak için şu modüler adımları takip etmelisin:

Veri Kümeleme (Data Partitioning):
02_preprocessing_and_pairs.ipynb notebook'unda veriyi tüm sinyallerin kalite skorlarını hesaplayarak şu üç klasöre ayır:

/data/train/high_quality/

/data/train/medium_quality/

/data/train/low_quality/ (Bunu eğitime alma, sadece test için sakla)

Siyam Ağlarını Eğitme:

Eğitim 1: Model A'yı sadece high_quality çiftleriyle eğit.

Eğitim 2: Model B'yi hem high_quality hem de medium_quality karışımıyla (Fine-tuning yaparak) eğit.

Performans Kıyaslaması:
Jüriye şunu göstereceksin: "Model A, yüksek kaliteli veride %99 doğruluk veriyor; Model B ise orta kaliteli veride %92 doğrulukla güvenliği korumaya devam ediyor."

"""

In [1]:
!pip install wfdb neurokit2 tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 688.9/688.9 kB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 111.1 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.


In [2]:
# Sistem ve Dosya Yönetimi
import os

# Veri Manipülasyonu ve Matematiksel İşlemler
import numpy as np
import pandas as pd

# Sinyal İşleme (EKG Temizleme ve Analiz)
import wfdb
import neurokit2 as nk

# İlerleme Çubuğu (Uzun döngüleri takip etmek için)
from tqdm import tqdm

# Görselleştirme (Ara kontroller için)
import matplotlib.pyplot as plt
import seaborn as sns

import itertools
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, as_completed

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
drive_base_path = '/content/drive/MyDrive/BeatKrypt/'
data_path = os.path.join(drive_base_path)
csv_path = os.path.join(data_path, 'ptbxl_database.csv')

QUALITY_HIGH_THRESHOLD = 0.75
QUALITY_MEDIUM_THRESHOLD = 0.50

# Örnekleme frekansı (Edge AI hedefi için)
sample_rate = 100

print(f"✓ Veri seti yolu: {csv_path}")

✓ Veri seti yolu: /content/drive/MyDrive/BeatKrypt/ptbxl_database.csv


In [5]:
ptbxl = pd.read_csv(csv_path)

In [6]:
# Her hastanın kaç kaydı olduğunu sayalım
patient_record_counts = ptbxl['patient_id'].value_counts().reset_index()
patient_record_counts.columns = ['patient_id', 'record_count']

# 3 veya daha fazla kaydı olan hastaları filtreleyelim
positive_patient_ids_df = patient_record_counts[patient_record_counts['record_count'] >= 3]

# Pozitif Çift Havuzu: 3 veya daha fazla kaydı olan hastaların ID'leri
positive_pair_patient_ids = positive_patient_ids_df['patient_id'].tolist()

# Negatif Çift Havuzu: Tüm benzersiz hasta ID'leri
negative_pair_patient_ids = ptbxl['patient_id'].unique().tolist()

print(f"Pozitif Çift Havuzu için Hasta ID sayısı: {len(positive_pair_patient_ids)}")
print(f"Negatif Çift Havuzu için Hasta ID sayısı: {len(negative_pair_patient_ids)}")

Pozitif Çift Havuzu için Hasta ID sayısı: 521
Negatif Çift Havuzu için Hasta ID sayısı: 18869


In [7]:
output_dir = os.path.join(drive_base_path, 'processed_data_split')
for tier in ['high', 'medium']:
    os.makedirs(os.path.join(output_dir, tier), exist_ok=True)

In [8]:
positive_pairs = []

pure_positives_df = ptbxl[ptbxl['patient_id'].isin(positive_pair_patient_ids)]

for patient_id in tqdm(positive_pair_patient_ids, desc="Pozitif Çiftler Oluşturuluyor"):
    patient_records = pure_positives_df[pure_positives_df['patient_id'] == patient_id]['filename_lr'].tolist()

    for rec1, rec2 in itertools.combinations(patient_records, 2):
        positive_pairs.append({
            'left_filename': rec1,
            'right_filename': rec2,
            'left_patient_id': patient_id,
            'right_patient_id': patient_id,   # pozitifte ikisi de aynı hasta
            'label': 1.0
        })

positive_pairs_df = pd.DataFrame(positive_pairs)
total_positive_count = len(positive_pairs_df)
print(f"\n✓ Toplam oluşturulan pozitif çift sayısı: {total_positive_count}")

Pozitif Çiftler Oluşturuluyor: 100%|██████████| 521/521 [00:00<00:00, 3078.94it/s]


✓ Toplam oluşturulan pozitif çift sayısı: 2684


In [9]:
negative_pairs = []

pure_negatives_df = ptbxl[ptbxl['patient_id'].isin(negative_pair_patient_ids)]
target_negative_count = total_positive_count

patient_to_files = pure_negatives_df.groupby('patient_id')['filename_lr'].apply(list).to_dict()
unique_patients = np.array(list(patient_to_files.keys()))

rng = np.random.default_rng(42)
left_patients = rng.choice(unique_patients, size=target_negative_count)

for left_patient in tqdm(left_patients, desc="Dengeli Negatif Çiftler Oluşturuluyor"):
    right_patient = rng.choice(unique_patients)
    while right_patient == left_patient:
        right_patient = rng.choice(unique_patients)

    left_filename = rng.choice(patient_to_files[left_patient])
    right_filename = rng.choice(patient_to_files[right_patient])

    negative_pairs.append({
        'left_filename': left_filename,
        'right_filename': right_filename,
        'left_patient_id': left_patient,
        'right_patient_id': right_patient,
        'label': 0.0
    })

negative_pairs_df = pd.DataFrame(negative_pairs)
print(f"\n✓ Toplam oluşturulan negatif çift sayısı: {len(negative_pairs_df)}")

Dengeli Negatif Çiftler Oluşturuluyor: 100%|██████████| 2684/2684 [00:00<00:00, 24216.06it/s]


✓ Toplam oluşturulan negatif çift sayısı: 2684


In [10]:
# 1. İki havuzu tek bir büyük DataFrame'de birleştir
all_pairs_df = pd.concat([positive_pairs_df, negative_pairs_df], ignore_index=True)

# 2. Satırları rastgele karıştır (frac=1 tüm veriyi al demek, random_state=42 ise tekrarlanabilirlik sağlar)
all_pairs_df = all_pairs_df.sample(frac=1, random_state=42).reset_index(drop=True)

# 3. Sonuçları doğrula
print(f"✓ Eşleştirme tablosu başarıyla birleştirildi ve karıştırıldı!")
print(f"Toplam Çift Sayısı: {len(all_pairs_df)}")
print(f"\nSınıf Dağılımı:\n{all_pairs_df['label'].value_counts()}")

# İlk 5 satıra göz atalım, 1.0 ve 0.0 etiketleri karışık gelmeli
all_pairs_df.head()

✓ Eşleştirme tablosu başarıyla birleştirildi ve karıştırıldı!
Toplam Çift Sayısı: 5368

Sınıf Dağılımı:
label
1.0    2684
0.0    2684
Name: count, dtype: int64


,left_filename,right_filename,left_patient_id,right_patient_id,label
0,records100/04000/04859_lr,records100/05000/05166_lr,17046.0,17046.0,1.0
1,records100/21000/21144_lr,records100/02000/02290_lr,8840.0,7699.0,0.0
2,records100/21000/21445_lr,records100/21000/21490_lr,18603.0,18603.0,1.0
3,records100/08000/08794_lr,records100/16000/16539_lr,13556.0,5799.0,0.0
4,records100/03000/03246_lr,records100/01000/01081_lr,1597.0,4456.0,0.0


In [11]:
BASE_DIR = drive_base_path

def process_single_row_with_quality(row_data):
    try:
        idx, row = row_data
        l_file = row['left_filename']
        r_file = row['right_filename']
        label = row['label']

        if not l_file.startswith('records100/'): l_file = os.path.join('records100', l_file)
        if not r_file.startswith('records100/'): r_file = os.path.join('records100', r_file)

        left_path = os.path.join(BASE_DIR, l_file)
        right_path = os.path.join(BASE_DIR, r_file)

        # Sinyal Okuma
        signal_left, _ = wfdb.rdsamp(left_path, channels=[0])
        signal_right, _ = wfdb.rdsamp(right_path, channels=[0])

        signal_left = signal_left.flatten()
        signal_right = signal_right.flatten()

        # Filtreleme
        clean_left = nk.ecg_clean(signal_left, sampling_rate=100, method="neurokit")
        clean_right = nk.ecg_clean(signal_right, sampling_rate=100, method="neurokit")

        if np.std(clean_left) < 0.01 or np.std(clean_right) < 0.01:
            return "low_variance", None

        # --- Kalite Skoru, peak detection'dan ÖNCE hesaplanıyor ---
        # Neden: rejected_quality olacak sinyaller için pahalı peak detection'ı boşa çalıştırmayalım
        q_left = nk.ecg_quality(clean_left, sampling_rate=100, method="averageQRS")
        q_right = nk.ecg_quality(clean_right, sampling_rate=100, method="averageQRS")

        # KRİTİK BUG FIX: NaN, tüm '<' karşılaştırmalarında False döner,
        # bu yüzden np.nan sessizce "high" grubuna sızabiliyordu. Erken yakala.
        if np.isnan(q_left) or np.isnan(q_right):
            return "rejected_quality", None

        avg_quality = (q_left + q_right) / 2.0

        if avg_quality < QUALITY_MEDIUM_THRESHOLD:
            return "rejected_quality", None
        elif avg_quality < QUALITY_HIGH_THRESHOLD:
            q_group = "medium"
        else:
            q_group = "high"

        # R-Tepe Bulma (artık sadece kaliteyi geçenler için çalışıyor)
        try:
            _, info_left = nk.ecg_peaks(clean_left, sampling_rate=100, method="neurokit")
            _, info_right = nk.ecg_peaks(clean_right, sampling_rate=100, method="neurokit")
            peaks_left = info_left["ECG_R_Peaks"]
            peaks_right = info_right["ECG_R_Peaks"]
        except Exception:
            return "no_peaks", None

        if len(peaks_left) == 0 or len(peaks_right) == 0:
            return "no_peaks", None

        # Segmentasyon ve Hizalama Kontrolü
        idx_l = next((p for p in peaks_left if 30 <= p < len(clean_left) - 50), None)
        idx_r = next((p for p in peaks_right if 30 <= p < len(clean_right) - 50), None)

        if idx_l is None or idx_r is None:
            return "no_peaks", None

        # Dilimleme
        slice_left = clean_left[idx_l - 30 : idx_l + 50]
        slice_right = clean_right[idx_r - 30 : idx_r + 50]

        # Normalizasyon
        norm_left = (slice_left - np.min(slice_left)) / (np.max(slice_left) - np.min(slice_left) + 1e-8)
        norm_right = (slice_right - np.min(slice_right)) / (np.max(slice_right) - np.min(slice_right) + 1e-8)

        # patient_id metadata eklendi -> train/test split'te leakage önlemek için şart
        return "success", (norm_left, norm_right, label, q_group,
                            row.get('left_patient_id'), row.get('right_patient_id'))

    except Exception:
        return "error", None

In [ ]:
X_high_left, X_high_right, y_high = [], [], []
X_med_left, X_med_right, y_med = [], [], []
meta_high, meta_med = [], []

status_counts = {"success": 0, "low_variance": 0, "no_peaks": 0, "rejected_quality": 0, "error": 0}

with ThreadPoolExecutor(max_workers=8) as executor:
    futures = [executor.submit(process_single_row_with_quality, row_data)
               for row_data in all_pairs_df.iterrows()]

    for future in tqdm(as_completed(futures), total=len(futures), desc="Kalite Bazlı İşleme"):
        status, result = future.result()
        status_counts[status] = status_counts.get(status, 0) + 1

        if status == "success":
            norm_left, norm_right, label, q_group, pid_left, pid_right = result
            if q_group == "high":
                X_high_left.append(norm_left)
                X_high_right.append(norm_right)
                y_high.append(label)
                meta_high.append({"patient_left": pid_left, "patient_right": pid_right, "label": label})
            elif q_group == "medium":
                X_med_left.append(norm_left)
                X_med_right.append(norm_right)
                y_med.append(label)
                meta_med.append({"patient_left": pid_left, "patient_right": pid_right, "label": label})

print("=== İŞLEME RAPORU ===")
for k, v in status_counts.items():
    print(f"{k}: {v}")
print(f"\n✓ High Quality: {len(y_high)} çift")
print(f"✓ Medium Quality: {len(y_med)} çift")

Kalite Bazlı İşleme:  23%|██▎       | 1215/5368 [03:24<08:29,  8.14it/s]/usr/local/lib/python3.12/dist-packages/neurokit2/signal/signal_period.py:83: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(
Kalite Bazlı İşleme:  44%|████▍     | 2371/5368 [05:47<05:05,  9.82it/s]

In [ ]:
OUTPUT_DIR = os.path.join(drive_base_path, 'processed_data_split')

for tier, (Xl, Xr, y_arr, meta) in {
    "high": (X_high_left, X_high_right, y_high, meta_high),
    "medium": (X_med_left, X_med_right, y_med, meta_med),
}.items():
    tier_dir = os.path.join(OUTPUT_DIR, tier)
    os.makedirs(tier_dir, exist_ok=True)
    np.save(os.path.join(tier_dir, f"X_{tier}_left.npy"), np.array(Xl, dtype=np.float32))
    np.save(os.path.join(tier_dir, f"X_{tier}_right.npy"), np.array(Xr, dtype=np.float32))
    np.save(os.path.join(tier_dir, f"y_{tier}.npy"), np.array(y_arr, dtype=np.float32))
    pd.DataFrame(meta).to_csv(os.path.join(tier_dir, f"metadata_{tier}.csv"), index=False)

    assert len(Xl) == len(Xr) == len(y_arr) == len(meta)
    print(f"✓ {tier.upper()} Quality Seti: {len(y_arr)} çift kaydedildi -> {tier_dir}")

In [ ]:
TIER_TO_VISUALIZE = "high"  # "high" veya "medium" seç

tier_dir = os.path.join(OUTPUT_DIR, TIER_TO_VISUALIZE)
X_left_loaded = np.load(os.path.join(tier_dir, f"X_{TIER_TO_VISUALIZE}_left.npy"))
X_right_loaded = np.load(os.path.join(tier_dir, f"X_{TIER_TO_VISUALIZE}_right.npy"))
y_loaded = np.load(os.path.join(tier_dir, f"y_{TIER_TO_VISUALIZE}.npy"))

print(f"=== VERİ DOĞRULAMA RAPORU ({TIER_TO_VISUALIZE.upper()}) ===")
print(f"Yüklenen X_left Şekli : {X_left_loaded.shape}")
print(f"Yüklenen X_right Şekli: {X_right_loaded.shape}")
print(f"Yüklenen y (Etiket) Şekli: {y_loaded.shape}")

random_idx = np.random.randint(0, len(y_loaded))
sample_left = X_left_loaded[random_idx]
sample_right = X_right_loaded[random_idx]
label = y_loaded[random_idx]

plt.figure(figsize=(12, 5))
plt.plot(sample_left, label='Sol Sinyal', color='#1f77b4', linewidth=2)
plt.plot(sample_right, label='Sağ Sinyal', color='#ff7f0e', linewidth=2, linestyle='--')
plt.axvline(x=30, color='red', linestyle=':', label='R-Tepe Hizası (İndeks 30)')
plt.title(f"[{TIER_TO_VISUALIZE.upper()}] Örnek Çift (İndeks: {random_idx}) - Etiket: {label} "
          f"({'Aynı Kişi' if label == 1.0 else 'Farklı Kişi'})")
plt.xlabel("Zaman / Örnek (Sample Index)")
plt.ylabel("Normalize Voltaj [0, 1]")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()